# WAFER-GUARD · Fab-7 Excursion Detector — Pre-Release Model Audit

**Role:** you are the final technical reviewer on the release board.

**Status:** the model below clears the release gate (**F1 ≥ 0.90**). Deployment to the line-monitoring service is scheduled for Friday 06:00.

**Time-box:** 120 minutes.

## Your mandate

A staff engineer has a bad feeling about this notebook and has asked you to sign off — or to stop the release.

1. **Audit.** Identify *every* defect that invalidates the reported metric or would degrade the deployed model. For each defect give: (a) exact location, (b) mechanism, (c) **reproducible evidence from your own runs** — numbers, shapes, ablations. Assertions without evidence earn no credit. You are not told how many defects exist.
2. **Repair.** Deliver a corrected end-to-end pipeline that respects production causality: *no information from time *t′ > t* (or from the evaluation population) may influence a prediction made at time *t*.*
3. **Re-evaluate.** Report honest metrics at the **natural defect prevalence (~6.6 %)**. Justify your metric(s) and your decision-threshold procedure.
4. **Write-up.** ≤ 1 page: findings ranked by severity, residual risks, and your go / no-go recommendation.

> Deliverables: a repaired notebook + `AUDIT.md`. The existing notebook runs top-to-bottom without error. That is not evidence that it is correct.

### Data dictionary — `secom_telemetry_modified.csv`

| column | description |
|---|---|
| `sensor_0 … sensor_589` | 590 in-line process / metrology signals per wafer lot (anonymised) |
| `label` | electrical-test outcome: `-1` = pass, `+1` = fail |
| `timestamp` | lot start time (the fab runs continuously; equipment ages, recipes drift) |

**Fab tool-matching note (from the process-integration team):** a fraction of wafers are routed through *Probe Station PS-L (legacy)*, whose firmware emits telemetry in a different format than the current-generation stations. Consult the data.


## 0 · Environment

In [51]:
!pip install "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib matplotlib

In [52]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, matthews_corrcoef)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0,
          train_frac=0.60, val_frac=0.20)

CODES = np.array([100., 200., 300.])
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.11.0+cpu | device=cpu


## 1. Data


In [53]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_raw = df[SENSOR_COLS].to_numpy(dtype=np.float64)
y_raw = (df["label"].to_numpy() == 1).astype(np.int64)

print(f"X: {X_raw.shape} | defect rate: {y_raw.mean():.2%} "
      f"({y_raw.sum()} fail / {(1 - y_raw).sum()} pass)")
print(f"missing cells: {np.isnan(X_raw).mean():.2%}")
print("time sorted:", df["timestamp"].is_monotonic_increasing)

X: (1567, 590) | defect rate: 6.64% (104 fail / 1463 pass)
missing cells: 4.54%
time sorted: True


## 2 · Preprocessing & class rebalancing


In [54]:
n = len(df)
i1 = int(HP["train_frac"] * n)
i2 = int((HP["train_frac"] + HP["val_frac"]) * n)
tr, va, te = slice(0, i1), slice(i1, i2), slice(i2, n)
for name, s in [("train", tr), ("val", va), ("test", te)]:
    print(name, y_raw[s].size, "pos", y_raw[s].sum(), f"{y_raw[s].mean():.2%}")

is_leg   = (X_raw[:, 0] == 0.0) | np.isin(X_raw[:, 1], CODES)
code_oh  = (X_raw[:, 1][:, None] == CODES[None, :])
unk_code = (is_leg & ~code_oh.any(axis=1))[:, None]
leg_feats = np.hstack([is_leg[:, None], code_oh, unk_code]).astype(np.float32)   # 5 cols

X_cont = X_raw.copy()
X_cont[is_leg, 0:2] = np.nan

imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
imputer.fit(X_cont[tr])
scaler = StandardScaler()
scaler.fit(imputer.transform(X_cont[tr]))

X_scaled = scaler.transform(imputer.transform(X_cont))
X_scaled = np.clip(X_scaled, -10, 10)
X_scaled[is_leg, 0:2] = 0.0
X_all = np.hstack([X_scaled, leg_feats]).astype(np.float32)

X_train_nat, y_train_nat = X_all[tr], y_raw[tr]
X_val,  y_val  = X_all[va], y_raw[va]
X_test, y_test = X_all[te], y_raw[te]

smote = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=SEED)
X_train, y_train = smote.fit_resample(X_train_nat, y_train_nat)
print(f"train {X_train_nat.shape} -> after SMOTE {X_train.shape} | defect rate: {y_train.mean():.2%}")
print(f"val {X_val.shape} | test {X_test.shape}")

train 940 pos 76 8.09%
val 313 pos 11 3.51%
test 314 pos 17 5.41%
train (940, 595) -> after SMOTE (1728, 595) | defect rate: 50.00%
val (313, 595) | test (314, 595)


## 3 · Tensors & loaders


In [55]:
F = X_train.shape[1]

def make_loader(X, y, shuffle):
    # keep an explicit channel axis -> [N, 1, F]  (so a Conv1d encoder can be dropped in later)
    Xt = torch.from_numpy(X).float().unsqueeze(1)
    yt = torch.from_numpy(y).float()
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(Xt, yt), batch_size=HP["batch_size"],
                      shuffle=shuffle, generator=g if shuffle else None, drop_last=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)
print(f"features: {F} | train batches: {len(train_loader)} | test batches: {len(test_loader)}")

features: 595 | train batches: 27 | test batches: 5


## 4 · Model — supervised autoencoder


In [56]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        x = x.flatten(start_dim=1)                       # tolerate [B, F] or [B, C, F]
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

model = WaferSAE(F, HP["hidden"], HP["latent"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 380,564


## 5 · Training


In [57]:
opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
bce   = nn.BCEWithLogitsLoss()

history = {"total": [], "recon": [], "clf": [], "val_ap": []}
best_ap, best_state = -1.0, None
xval = torch.from_numpy(X_val).float().to(DEVICE)

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    acc = dict(total=0.0, recon=0.0, clf=0.0); n_batches = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        reconstructed, logits, _ = model(batch_x)

        target = batch_x.flatten(start_dim=1)
        assert reconstructed.shape == target.shape, (reconstructed.shape, target.shape)
        assert logits.shape == batch_y.shape, (logits.shape, batch_y.shape)

        recon_loss = torch.mean((reconstructed - target) ** 2)
        clf_loss   = bce(logits, batch_y)
        loss       = recon_loss + HP["lambda_clf"] * clf_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        acc["total"] += loss.item(); acc["recon"] += recon_loss.item(); acc["clf"] += clf_loss.item()
        n_batches += 1

    sched.step()
    for k in ("total", "recon", "clf"):
        history[k].append(acc[k] / n_batches)

    model.eval()
    with torch.no_grad():
        pv = torch.sigmoid(model(xval)[1]).cpu().numpy()
    val_ap = average_precision_score(y_val, pv)
    history["val_ap"].append(val_ap)
    if val_ap > best_ap:
        best_ap = val_ap
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d}/{HP['epochs']} | total {history['total'][-1]:.4f} "
              f"| recon {history['recon'][-1]:.4f} | clf {history['clf'][-1]:.4f} | val AP {val_ap:.4f}")

model.load_state_dict(best_state)
print("best val AP:", round(best_ap, 4))

epoch 01/40 | total 0.9146 | recon 0.5814 | clf 0.3332 | val AP 0.0303
epoch 05/40 | total 0.4189 | recon 0.4171 | clf 0.0018 | val AP 0.0311
epoch 10/40 | total 0.3119 | recon 0.3108 | clf 0.0011 | val AP 0.0322
epoch 15/40 | total 0.2583 | recon 0.2577 | clf 0.0006 | val AP 0.0314
epoch 20/40 | total 0.2304 | recon 0.2299 | clf 0.0005 | val AP 0.0308
epoch 25/40 | total 0.2147 | recon 0.2143 | clf 0.0004 | val AP 0.0317
epoch 30/40 | total 0.2052 | recon 0.2049 | clf 0.0003 | val AP 0.0302
epoch 35/40 | total 0.2012 | recon 0.2008 | clf 0.0004 | val AP 0.0305
epoch 40/40 | total 0.1995 | recon 0.1993 | clf 0.0003 | val AP 0.0305
best val AP: 0.033


## 6 · Evaluation


In [58]:
model.eval()
with torch.no_grad():
    p_val  = torch.sigmoid(model(torch.from_numpy(X_val).float().to(DEVICE))[1]).cpu().numpy()
    p_test = torch.sigmoid(model(torch.from_numpy(X_test).float().to(DEVICE))[1]).cpu().numpy()

cands = np.unique(p_val)
f1s = [f1_score(y_val, p_val >= t, zero_division=0) for t in cands]
thr = cands[int(np.argmax(f1s))]
preds = (p_test >= thr).astype(int)

f1   = f1_score(y_test, preds, zero_division=0)
prec = precision_score(y_test, preds, zero_division=0)
rec  = recall_score(y_test, preds)
mcc  = matthews_corrcoef(y_test, preds)
auc  = roc_auc_score(y_test, p_test)
ap   = average_precision_score(y_test, p_test)
tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

print(f"threshold {thr:.4f} (from val) | test prevalence {y_test.mean():.2%}")
print(f"F1 {f1:.4f} | precision {prec:.4f} | recall {rec:.4f} | MCC {mcc:.4f}")
print(f"ROC-AUC {auc:.4f} | PR-AUC {ap:.4f} (baseline = {y_test.mean():.4f})")
print(f"confusion  TN={tn}  FP={fp}  FN={fn}  TP={tp}")

rng = np.random.default_rng(SEED)
boot_f1, boot_ap = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_test), len(y_test))
    if y_test[idx].sum() > 0:
        boot_f1.append(f1_score(y_test[idx], p_test[idx] >= thr, zero_division=0))
        boot_ap.append(average_precision_score(y_test[idx], p_test[idx]))
print("F1 95% CI", np.percentile(boot_f1, [2.5, 97.5]).round(3))
print("PR-AUC 95% CI", np.percentile(boot_ap, [2.5, 97.5]).round(3))


threshold 0.0009 (from val) | test prevalence 5.41%
F1 0.1067 | precision 0.0577 | recall 0.7059 | MCC 0.0220
ROC-AUC 0.5310 | PR-AUC 0.0734 (baseline = 0.0541)
confusion  TN=101  FP=196  FN=5  TP=12
F1 95% CI [0.056 0.164]
PR-AUC 95% CI [0.039 0.163]


In [59]:
print("buggy shape:", (torch.randn(64, 590) - torch.randn(64, 1, 590)).shape)

print(f"legacy n={is_leg.sum()} fail rate={y_raw[is_leg].mean():.4f} | "
      f"non-legacy n={(~is_leg).sum()} fail rate={y_raw[~is_leg].mean():.4f}")
for name, s in [("train", tr), ("val", va), ("test", te)]:
    print(name, "legacy:", is_leg[s].sum(), "pos in legacy:", y_raw[s][is_leg[s]].sum(),
          "| non-legacy:", (~is_leg[s]).sum(), "pos:", y_raw[s][~is_leg[s]].sum())

P = {"SAE": p_test}
lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000).fit(X_train_nat, y_train_nat)
rf = RandomForestClassifier(500, min_samples_leaf=5, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=SEED).fit(X_train_nat, y_train_nat)
P["LR"] = lr.predict_proba(X_test)[:, 1]
P["RF"] = rf.predict_proba(X_test)[:, 1]

lr_ns = LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000).fit(X_train_nat[:, :-5], y_train_nat)
P["LR-no-routing"] = lr_ns.predict_proba(X_test[:, :-5])[:, 1]

nl = ~is_leg[te]
rows = []
for name, p in P.items():
    rows.append((name, "all", len(y_test), y_test.sum(), roc_auc_score(y_test, p),
                 average_precision_score(y_test, p), y_test.mean()))
    if name != "LR-no-routing":
        rows.append((name, "non-legacy", nl.sum(), y_test[nl].sum(), roc_auc_score(y_test[nl], p[nl]),
                     average_precision_score(y_test[nl], p[nl]), y_test[nl].mean()))
print(pd.DataFrame(rows, columns=["model", "population", "n", "pos", "ROC_AUC", "PR_AUC", "baseline_AP"])
        .round(3).to_string(index=False))

buggy shape: torch.Size([64, 64, 590])
legacy n=146 fail rate=0.0000 | non-legacy n=1421 fail rate=0.0732
train legacy: 79 pos in legacy: 0 | non-legacy: 861 pos: 76
val legacy: 40 pos in legacy: 0 | non-legacy: 273 pos: 11
test legacy: 27 pos in legacy: 0 | non-legacy: 287 pos: 17
        model population   n  pos  ROC_AUC  PR_AUC  baseline_AP
          SAE        all 314   17    0.531   0.073        0.054
          SAE non-legacy 287   17    0.528   0.078        0.059
           LR        all 314   17    0.673   0.098        0.054
           LR non-legacy 287   17    0.665   0.104        0.059
           RF        all 314   17    0.582   0.082        0.054
           RF non-legacy 287   17    0.573   0.087        0.059
LR-no-routing        all 314   17    0.672   0.098        0.054


In [60]:
mask_df = pd.read_csv("secom_legacy_mask_EVALUATOR_ONLY.csv", parse_dates=["timestamp"])
orig = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])   # original row order
Xo = orig[SENSOR_COLS].to_numpy(dtype=np.float64)
found = (Xo[:, 0] == 0.0) | np.isin(Xo[:, 1], CODES)
truth = np.zeros(len(orig), dtype=bool)
truth[mask_df["row"].to_numpy()] = True

print("detected", found.sum(), "| true", truth.sum())
print("TP", (found & truth).sum(), "FP", (found & ~truth).sum(), "FN", (~found & truth).sum())
print("codes match:", (orig.loc[mask_df["row"], "sensor_1"].to_numpy() == mask_df["sensor_1_code"].to_numpy()).mean())

detected 146 | true 146
TP 146 FP 0 FN 0
codes match: 1.0



## 7 · Release gate



In [61]:
GATE_LIFT, GATE_REC, GATE_PREC = 2.0, 0.5, 0.2
lift = ap / y_test.mean()
passed = (lift >= GATE_LIFT) and (rec >= GATE_REC) and (prec >= GATE_PREC)
print(f"PR-AUC lift {lift:.2f}x | recall {rec:.2f} | precision {prec:.2f}")
print("Release gate:", "GO" if passed else "NO-GO")

os.makedirs("artifacts", exist_ok=True)
tag = "candidate" if passed else "REJECTED_do_not_deploy"
torch.save(model.state_dict(), f"artifacts/wafer_sae_{tag}.pt")
joblib.dump({"imputer": imputer, "scaler": scaler, "codes": CODES,
             "threshold": float(thr), "gate_passed": bool(passed)},
            f"artifacts/preprocessor_{tag}.joblib")
print("saved for reference only, NOT approved for deployment" if not passed else "saved as release candidate")

PR-AUC lift 1.35x | recall 0.71 | precision 0.06
Release gate: NO-GO
saved for reference only, NOT approved for deployment
